<a href="https://colab.research.google.com/github/Vdivyajaswanthi123/ai-mentor-portfolio/blob/main/Day7_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q chromadb sentence-transformers scikit-learn matplotlib

from sentence_transformers import SentenceTransformer
from chromadb import PersistentClient

# Free, local, 384-dim embeddings
embed = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('Embedding dim:', embed.get_sentence_embedding_dimension())

In [ ]:
with open('./data/syllabi_cached/cse_sem5.txt') as f:
    text = f.read()

# Split on blank lines into paragraphs
paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()][:10]

print(f'Loaded {len(paragraphs)} paragraphs')
for i, p in enumerate(paragraphs):
    print(f'  [{i+1}] {p[:80]}')

In [ ]:
client = PersistentClient(path='./chroma_db')
col = client.get_or_create_collection('hello_syllabus')

# Embed all 10 paragraphs
vectors = embed.encode(paragraphs).tolist()

# Add to collection (with sequential IDs)
col.add(
    documents=paragraphs,
    embeddings=vectors,
    ids=[f'p{i}' for i in range(len(paragraphs))]
)

print(f'Indexed {col.count()} documents')

In [ ]:
queries = [
    'what is dynamic programming?',
    'machine learning topics',
    'operating system processes',
]

for q in queries:
    print(f'\nQuery: {q}')
    qv = embed.encode([q]).tolist()
    results = col.query(query_embeddings=qv, n_results=3)
    docs = results['documents'][0]
    distances = results['distances'][0]
    for j, (d, dist) in enumerate(zip(docs, distances)):
        print(f'  [{j+1}] (dist={dist:.3f}) {d[:80]}')

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np

vectors_arr = np.array(vectors)

pca = PCA(n_components=2)
xy = pca.fit_transform(vectors_arr)

plt.figure(figsize=(10, 8))
plt.scatter(xy[:, 0], xy[:, 1], s=100, alpha=0.6)
for i, p in enumerate(paragraphs):
    label = p[:30] + '...' if len(p) > 30 else p
    plt.annotate(label, (xy[i, 0], xy[i, 1]), fontsize=8)
plt.title('Syllabus Paragraph Embeddings (PCA 2D)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Add a totally unrelated paragraph to see where it lands
outlier = "Today's special at the cafeteria is butter chicken with rice and naan."

col.add(
    documents=[outlier],
    embeddings=embed.encode([outlier]).tolist(),
    ids=['outlier_food']
)

# Re-fetch all and re-plot
all_docs = col.get(include=['embeddings', 'documents'])
all_vecs = np.array(all_docs['embeddings'])
labels = all_docs['ids']

pca = PCA(n_components=2) # Re-initialize PCA to ensure it fits the new data
xy = pca.fit_transform(all_vecs)
plt.figure(figsize=(10, 8))
colors = ['red' if 'outlier' in l else 'blue' for l in labels]
plt.scatter(xy[:, 0], xy[:, 1], c=colors, s=100, alpha=0.6)
for i, l in enumerate(labels):
    short = labels[i] if 'outlier' in labels[i] else all_docs['documents'][i][:30] + '...'
    plt.annotate(short, (xy[i, 0], xy[i, 1]), fontsize=8)
plt.title('With outlier (red)')
plt.show()